# My Voice Recording Transcription + TTS Dataset Builder

Use this notebook on Kaggle for the clean recordings in `audio/my_voice_recordings`.

## Before running

1. Upload `my_voice_recordings_upload.zip` as a Kaggle Dataset, or upload the folder containing `manifest.jsonl` and `normalized/`.
2. Add that dataset to this notebook.
3. Enable Internet.
4. Use GPU T4 if available.
5. Run cells in order.

This notebook does **not** use diarization. These are single-speaker recordings.

In [ ]:
# CELL 1 - Install dependencies
!pip install -q faster-whisper soundfile pandas tqdm

In [ ]:
# CELL 2 - Locate uploaded dataset
import json, os, shutil, subprocess, zipfile
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/my_voice_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def find_manifest():
    manifests = sorted(INPUT_ROOT.rglob('manifest.jsonl'))
    if manifests:
        return manifests[0]

    zips = sorted(INPUT_ROOT.rglob('*.zip'))
    for z in zips:
        extract_dir = WORK_ROOT / 'uploaded_zip'
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z, 'r') as handle:
            handle.extractall(extract_dir)
        manifests = sorted(extract_dir.rglob('manifest.jsonl'))
        if manifests:
            return manifests[0]
    raise FileNotFoundError('Could not find manifest.jsonl in /kaggle/input. Upload my_voice_recordings_upload.zip or the folder.')

MANIFEST = find_manifest()
DATA_ROOT = MANIFEST.parent
print('Manifest:', MANIFEST)
print('Data root:', DATA_ROOT)

rows = []
with open(MANIFEST, 'r', encoding='utf-8-sig') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print('Rows:', len(rows))
pd.DataFrame(rows)[['speaker', 'language', 'duration', 'audio_path']].head()

In [ ]:
# CELL 3 - Fix audio paths for Kaggle filesystem
def resolve_audio_path(row):
    original = Path(row['audio_path'])
    name = original.name
    candidates = list(DATA_ROOT.rglob(name))
    if candidates:
        return candidates[0]
    candidates = list(INPUT_ROOT.rglob(name))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f'Could not find audio file for {name}')

for row in rows:
    row['kaggle_audio_path'] = str(resolve_audio_path(row))

df = pd.DataFrame(rows)
display(df.groupby(['speaker', 'language'])['duration'].agg(['count', 'sum']))
print('Total minutes:', round(df['duration'].sum() / 60, 2))

In [ ]:
# CELL 4 - Load faster-whisper model
from faster_whisper import WhisperModel

# Recommended:
# - base: fastest, lower quality
# - medium: good balance on Kaggle T4
# - large-v3: best quality, slower/heavier
MODEL_SIZE = 'medium'
DEVICE = 'cuda'
COMPUTE_TYPE = 'float16'

model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print('Loaded', MODEL_SIZE)

In [ ]:
# CELL 5 - Transcribe files
TRANSCRIPTS_DIR = Path('/kaggle/working/my_voice_transcripts')
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

LANG_HINTS = {
    'english': 'en',
    'hindi': 'hi',
    'tamil': 'ta',
    # Bhojpuri is not a first-class Whisper language. Hindi hint often works better than auto.
    'bhojpuri': 'hi',
    'mixed': None,
}

transcript_rows = []

for row in tqdm(rows):
    audio_path = Path(row['kaggle_audio_path'])
    stem = audio_path.stem
    lang = LANG_HINTS.get(row.get('language'), None)
    json_path = TRANSCRIPTS_DIR / f'{stem}.json'
    txt_path = TRANSCRIPTS_DIR / f'{stem}.txt'

    if json_path.exists():
        payload = json.loads(json_path.read_text(encoding='utf-8'))
    else:
        segments, info = model.transcribe(
            str(audio_path),
            language=lang,
            beam_size=5,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=450),
        )
        segs = []
        for s in segments:
            segs.append({
                'start': round(float(s.start), 3),
                'end': round(float(s.end), 3),
                'text': s.text.strip(),
            })
        text = ' '.join(s['text'] for s in segs).strip()
        payload = {
            **row,
            'detected_language': getattr(info, 'language', None),
            'language_probability': float(getattr(info, 'language_probability', 0.0) or 0.0),
            'model': MODEL_SIZE,
            'segments': segs,
            'text': text,
        }
        json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
        txt_path.write_text(text + '\n', encoding='utf-8')

    transcript_rows.append({**row, 'transcript_json': str(json_path), 'transcript_txt': str(txt_path), 'draft_text': payload.get('text', '')})

transcript_manifest = Path('/kaggle/working/transcripts_manifest.jsonl')
with open(transcript_manifest, 'w', encoding='utf-8') as f:
    for row in transcript_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Wrote', transcript_manifest)

In [ ]:
# CELL 6 - Build segmented draft TTS dataset
DATASET_DIR = Path('/kaggle/working/my_voice_dataset')
WAVS_DIR = DATASET_DIR / 'wavs'
WAVS_DIR.mkdir(parents=True, exist_ok=True)

MIN_SEC = 2.0
MAX_SEC = 14.0
MIN_CHARS = 8
PAD_SEC = 0.08

def ffmpeg_slice(src, dst, start, end):
    start = max(0.0, float(start) - PAD_SEC)
    duration = max(0.1, float(end) - start + PAD_SEC)
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-ss', f'{start:.3f}', '-t', f'{duration:.3f}',
        '-i', str(src),
        '-ac', '1', '-ar', '24000', '-sample_fmt', 's16',
        str(dst),
    ]
    subprocess.run(cmd, check=True)

dataset_rows = []
counter = 0

for row in tqdm(transcript_rows):
    payload = json.loads(Path(row['transcript_json']).read_text(encoding='utf-8'))
    src = Path(row['kaggle_audio_path'])
    for seg in payload.get('segments', []):
        text = seg.get('text', '').strip()
        dur = float(seg['end']) - float(seg['start'])
        if dur < MIN_SEC or dur > MAX_SEC:
            continue
        if len(text) < MIN_CHARS:
            continue
        out_name = f"{row['speaker']}_{row['language']}_{counter:05d}.wav"
        out_path = WAVS_DIR / out_name
        ffmpeg_slice(src, out_path, seg['start'], seg['end'])
        dataset_rows.append({
            'audio_path': str(out_path),
            'text': text,
            'speaker': row['speaker'],
            'language': row['language'],
            'source_file': src.name,
            'start': seg['start'],
            'end': seg['end'],
            'duration': round(dur, 3),
            'needs_review': True,
        })
        counter += 1

dataset_jsonl = DATASET_DIR / 'dataset.draft.jsonl'
metadata_csv = DATASET_DIR / 'metadata.draft.csv'

with open(dataset_jsonl, 'w', encoding='utf-8') as f:
    for row in dataset_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

pd.DataFrame(dataset_rows).to_csv(metadata_csv, index=False)

print('Segments:', len(dataset_rows))
print('Total minutes:', round(sum(r['duration'] for r in dataset_rows) / 60, 2))
print('Dataset:', DATASET_DIR)

In [ ]:
# CELL 7 - Preview summary
draft = pd.DataFrame(dataset_rows)
if len(draft):
    display(draft.groupby(['speaker', 'language'])['duration'].agg(['count', 'sum']))
    display(draft.sample(min(10, len(draft)), random_state=1)[['speaker', 'language', 'duration', 'text']])
else:
    print('No segments produced. Check transcription output and filters.')

In [ ]:
# CELL 8 - Zip outputs for download
!cd /kaggle/working && zip -qr my_voice_transcripts.zip my_voice_transcripts transcripts_manifest.jsonl
!cd /kaggle/working && zip -qr my_voice_dataset_draft.zip my_voice_dataset
print('/kaggle/working/my_voice_transcripts.zip')
print('/kaggle/working/my_voice_dataset_draft.zip')